# Sentinel-2 HR Dataset Preparation + Qwen Captioning

This notebook extracts **only HR Sentinel-2 RGB patches** from `.SAFE.zip` products in Google Drive. GeoDiff-GAN does **not** need LR files saved on disk; synthetic 40 m LR is generated during training from each HR patch.

Output layout:

```text
geodiff_hr_dataset/
  HR/<city_name>/*.npz
  metadata/products/*.jsonl
  metadata/products/*.done
  metadata/summary.csv
  manifest_relative.jsonl
  captions_qwen3vl.jsonl      # optional
```

Captioning creates three variants per patch: `brief`, `descriptive`, and `analytical`. The top-level `caption` field remains backward-compatible with GeoDiff-GAN training.

In [ ]:
!pip install -q rasterio tqdm pandas pillow

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

INPUT_ZIP_DIR = Path('/content/drive/MyDrive/thesis/sentinel2')
OUTPUT_ROOT = Path('/content/drive/MyDrive/thesis/geodiff_hr_dataset')
HR_ROOT = OUTPUT_ROOT / 'HR'
META_ROOT = OUTPUT_ROOT / 'metadata'
PRODUCT_META_ROOT = META_ROOT / 'products'

PATCH_SIZE = 512
STRIDE = 384
MINIMUM_VALID_FRACTION = 0.95
REFLECTANCE_SCALE = 10000.0
SATURATION_VALUE = 1.0

BLACK_THRESHOLD = 0.003
MAX_BLACK_FRACTION = 0.02
MAX_ZERO_FRACTION = 0.001

VAL_PREFIXES = ['CHHATARPUR2']
TEST_PREFIXES = ['CHHATARPUR1']
UNMATCHED_SPLIT = 'train'

# Leave None for full incremental processing. New .SAFE.zip files added later are included automatically.
MAX_PRODUCTS = None

# Keep False for restart safety: existing patches are skipped and missing windows continue.
OVERWRITE_PATCHES = False

# Display only. Sentinel-2 reflectance is dark in natural RGB without gain.
VISUAL_DISPLAY_GAIN = 2.5

HR_ROOT.mkdir(parents=True, exist_ok=True)
META_ROOT.mkdir(parents=True, exist_ok=True)
PRODUCT_META_ROOT.mkdir(parents=True, exist_ok=True)
print('Input:', INPUT_ZIP_DIR)
print('Output:', OUTPUT_ROOT)


In [ ]:
import json, re, shutil, zipfile
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio
from rasterio.enums import Resampling
from rasterio.windows import Window, bounds, from_bounds
from tqdm.auto import tqdm

INVALID_SCL_CLASSES = {0, 1, 3, 8, 9, 10, 11}
SENTINEL_PRODUCT_PATTERN = re.compile(r'(S2[A-Z]_MSIL2A_[A-Z0-9_]+\.SAFE)$', re.I)

@dataclass
class ManifestRecord:
    patch: str
    tile_id: str
    split: str
    row: int
    col: int
    valid_fraction: float
    source: str = 'copernicus_sentinel2_l2a'
    license_id: str = 'copernicus-free-full-open'
    caption: str = ''
    source_product: str = ''

def city_from_zip_name(path: Path) -> str:
    name = re.sub(r'\.zip$', '', path.name, flags=re.I)
    name = re.sub(r'\.SAFE$', '', name, flags=re.I)
    match = re.match(r'(.+?)_S2[A-Z]_MSIL2A_', name, flags=re.I)
    return match.group(1) if match else name.split('_')[0]

def canonical_product_id(product: str | Path) -> str:
    name = Path(product).name
    match = SENTINEL_PRODUCT_PATTERN.search(name)
    return match.group(1) if match else name

def is_safe_product_root(product: str | Path) -> bool:
    product = Path(product)
    return product.is_dir() and (product / 'manifest.safe').is_file() and (product / 'GRANULE').is_dir()

def discover_safe_products(root: str | Path) -> list[Path]:
    root = Path(root)
    candidates = list(root.rglob('*.SAFE'))
    if root.suffix.casefold() == '.safe':
        candidates.insert(0, root)
    products = {}
    for candidate in sorted(set(candidates)):
        if is_safe_product_root(candidate):
            products.setdefault(canonical_product_id(candidate).casefold(), candidate)
    return [products[key] for key in sorted(products)]

def tile_id_from_product(product: Path) -> str:
    match = re.search(r'_T([0-9]{2}[A-Z]{3})_', product.name)
    if match:
        return match.group(1)
    granules = list((product / 'GRANULE').glob('*')) if (product / 'GRANULE').exists() else []
    for granule in granules:
        match = re.search(r'_T([0-9]{2}[A-Z]{3})_', granule.name)
        if match:
            return match.group(1)
    return product.stem

def find_band(product: Path, pattern: str) -> Path:
    matches = sorted(product.rglob(pattern))
    if not matches:
        raise FileNotFoundError(f'Could not find {pattern} below {product}')
    return matches[0]

def product_split(source_product: str) -> str:
    lower = source_product.lower()
    if any(lower.startswith(prefix.lower()) for prefix in VAL_PREFIXES):
        return 'val'
    if any(lower.startswith(prefix.lower()) for prefix in TEST_PREFIXES):
        return 'test'
    return UNMATCHED_SPLIT

def record_to_json(record: ManifestRecord) -> str:
    return json.dumps(asdict(record), ensure_ascii=True)

def load_records(path: Path) -> list[ManifestRecord]:
    if not path.exists():
        return []
    records = []
    for line in path.read_text(encoding='utf-8').splitlines():
        if line.strip():
            records.append(ManifestRecord(**json.loads(line)))
    return records

def dedupe_records(records: list[ManifestRecord]) -> list[ManifestRecord]:
    by_patch = {}
    for record in records:
        by_patch[record.patch] = record
    return [by_patch[key] for key in sorted(by_patch)]

def write_jsonl(path: Path, records: list[ManifestRecord]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8') as handle:
        for record in dedupe_records(records):
            handle.write(record_to_json(record) + '\n')

def append_partial_record(handle, record: ManifestRecord) -> None:
    handle.write(record_to_json(record) + '\n')
    handle.flush()

def extract_one_safe_product(
    product: Path,
    city: str,
    source_product: str,
    existing_records: list[ManifestRecord],
    partial_handle,
) -> list[ManifestRecord]:
    tile_id = tile_id_from_product(product)
    split = product_split(source_product)
    city_dir = HR_ROOT / city
    city_dir.mkdir(parents=True, exist_ok=True)

    existing_by_patch = {record.patch: record for record in existing_records}
    emitted_this_call = set()

    red_path = find_band(product, '*_B04_10m.jp2')
    green_path = find_band(product, '*_B03_10m.jp2')
    blue_path = find_band(product, '*_B02_10m.jp2')
    scl_path = find_band(product, '*_SCL_20m.jp2')
    records = []

    with rasterio.open(red_path) as red, rasterio.open(green_path) as green, rasterio.open(blue_path) as blue, rasterio.open(scl_path) as scl:
        height, width = red.height, red.width
        rows = list(range(0, max(height - PATCH_SIZE + 1, 1), STRIDE))
        for row in tqdm(rows, desc=f'{city} {tile_id}', leave=False):
            for col in range(0, max(width - PATCH_SIZE + 1, 1), STRIDE):
                if row + PATCH_SIZE > height or col + PATCH_SIZE > width:
                    continue

                patch_name = f'{city}_{tile_id}_{Path(source_product).stem}_r{row:05d}_c{col:05d}.npz'
                patch_path = city_dir / patch_name
                relative_patch = patch_path.relative_to(OUTPUT_ROOT).as_posix()

                if relative_patch in existing_by_patch and patch_path.exists():
                    records.append(existing_by_patch[relative_patch])
                    continue

                if patch_path.exists() and not OVERWRITE_PATCHES:
                    # Crash recovery: patch exists but partial JSONL did not record it yet.
                    record = ManifestRecord(relative_patch, tile_id, split, row, col, 1.0, source_product=source_product)
                    records.append(record)
                    existing_by_patch[relative_patch] = record
                    if relative_patch not in emitted_this_call:
                        append_partial_record(partial_handle, record)
                        emitted_this_call.add(relative_patch)
                    continue

                window = Window(col, row, PATCH_SIZE, PATCH_SIZE)
                rgb = np.stack([red.read(1, window=window), green.read(1, window=window), blue.read(1, window=window)]).astype(np.float32)
                scl_window = from_bounds(*bounds(window, red.transform), transform=scl.transform)
                scl_values = scl.read(1, window=scl_window, out_shape=(PATCH_SIZE, PATCH_SIZE), resampling=Resampling.nearest, boundless=True, fill_value=0)

                valid = ~np.isin(scl_values, list(INVALID_SCL_CLASSES))
                valid &= np.isfinite(rgb).all(axis=0)
                valid &= (rgb > 0).all(axis=0)
                valid &= (rgb < REFLECTANCE_SCALE * SATURATION_VALUE).all(axis=0)
                valid_fraction = float(valid.mean())
                if valid_fraction < MINIMUM_VALID_FRACTION:
                    continue

                hr = np.clip(rgb / REFLECTANCE_SCALE, 0, 1).astype(np.float32)
                zero_fraction = float((rgb <= 0).any(axis=0).mean())
                black_fraction = float((hr < BLACK_THRESHOLD).all(axis=0).mean())
                if zero_fraction > MAX_ZERO_FRACTION or black_fraction > MAX_BLACK_FRACTION:
                    continue

                np.savez_compressed(patch_path, hr=hr, valid_mask=valid.astype(np.uint8), transform=np.asarray(red.window_transform(window))[:2].reshape(-1), crs=str(red.crs))
                record = ManifestRecord(relative_patch, tile_id, split, row, col, valid_fraction, source_product=source_product)
                records.append(record)
                existing_by_patch[relative_patch] = record
                append_partial_record(partial_handle, record)
                emitted_this_call.add(relative_patch)
    return records

def process_zip(zip_path: Path) -> list[ManifestRecord]:
    city = city_from_zip_name(zip_path)
    source_product = zip_path.name.replace('.zip', '')
    record_path = PRODUCT_META_ROOT / f'{source_product}.jsonl'
    partial_path = PRODUCT_META_ROOT / f'{source_product}.partial.jsonl'
    done_path = PRODUCT_META_ROOT / f'{source_product}.done'

    if done_path.exists() and record_path.exists():
        print('Skipping completed:', source_product)
        return load_records(record_path)

    existing_records = dedupe_records(load_records(partial_path) + load_records(record_path))
    if existing_records:
        print(f'Resuming incomplete product: {source_product} with {len(existing_records)} recorded patches')

    extract_root = Path('/content/safe_extract')
    if extract_root.exists():
        shutil.rmtree(extract_root)
    extract_root.mkdir(parents=True, exist_ok=True)

    print('Extracting:', zip_path.name)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(extract_root)

    products = discover_safe_products(extract_root)
    if not products:
        raise RuntimeError(f'No valid SAFE product found inside {zip_path}')

    all_records = list(existing_records)
    with partial_path.open('a', encoding='utf-8') as partial_handle:
        for product in products:
            all_records.extend(
                extract_one_safe_product(
                    product,
                    city,
                    source_product,
                    dedupe_records(all_records),
                    partial_handle,
                )
            )

    final_records = dedupe_records(all_records)
    write_jsonl(record_path, final_records)
    done_path.write_text('done\n', encoding='utf-8')
    if partial_path.exists():
        partial_path.unlink()
    shutil.rmtree(extract_root)
    print(f'Completed {source_product}: {len(final_records)} patches')
    return final_records

def collect_all_product_records(include_partials: bool = True) -> list[ManifestRecord]:
    records = []
    for path in sorted(PRODUCT_META_ROOT.glob('*.jsonl')):
        if path.name.endswith('.partial.jsonl') and not include_partials:
            continue
        records.extend(load_records(path))
    return dedupe_records(records)


In [ ]:
zip_files = sorted(INPUT_ZIP_DIR.glob('*.SAFE.zip'))
if MAX_PRODUCTS is not None:
    zip_files = zip_files[:MAX_PRODUCTS]

print(f'Found {len(zip_files)} zip files')
for path in zip_files[:10]:
    print(' -', path.name)

# New zip files added later are processed automatically; completed products are skipped.
for zip_path in tqdm(zip_files, desc='SAFE zips'):
    process_zip(zip_path)

all_records = collect_all_product_records(include_partials=True)
manifest_path = OUTPUT_ROOT / 'manifest_relative.jsonl'
write_jsonl(manifest_path, all_records)
summary = pd.DataFrame([asdict(record) for record in all_records])
summary_path = OUTPUT_ROOT / 'metadata' / 'summary.csv'
summary.to_csv(summary_path, index=False)

print('Manifest:', manifest_path)
print('Summary:', summary_path)
print('Total patches:', len(summary))
if len(summary):
    display(summary.groupby(['split', 'tile_id']).size().reset_index(name='patches'))
    display(summary.groupby(['source_product']).size().reset_index(name='patches'))


In [ ]:
import random
import matplotlib.pyplot as plt

manifest_path = OUTPUT_ROOT / 'manifest_relative.jsonl'
records = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]
summary = pd.DataFrame(records)

print('Patch count:', len(records))
if len(summary):
    display(summary.groupby('split').size().reset_index(name='patches'))
    display(summary.groupby(summary['patch'].map(lambda p: Path(p).parts[1] if len(Path(p).parts) > 1 else 'unknown')).size().reset_index(name='city_patches'))

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    summary['split'].value_counts().plot(kind='bar', ax=axes[0], title='Patches by split')
    summary['patch'].map(lambda p: Path(p).parts[1] if len(Path(p).parts) > 1 else 'unknown').value_counts().head(20).plot(kind='bar', ax=axes[1], title='Top cities/products')
    plt.tight_layout()
    plt.show()

samples = random.sample(records, min(8, len(records)))
cols = 4
rows = max(1, int(np.ceil(len(samples) / cols)))
plt.figure(figsize=(4 * cols, 4 * rows))
for i, record in enumerate(samples, 1):
    patch_path = OUTPUT_ROOT / record['patch']
    with np.load(patch_path) as data:
        hr = data['hr']
        valid_mask = data.get('valid_mask')
    image = np.transpose(hr, (1, 2, 0)) if hr.shape[0] == 3 else hr
    plt.subplot(rows, cols, i)
    plt.imshow(np.clip(image * VISUAL_DISPLAY_GAIN, 0, 1))
    title = f"{record['split']} | {record['tile_id']}\n{Path(record['patch']).parent.name}"
    if valid_mask is not None:
        title += f" | valid={valid_mask.mean():.2f}"
    plt.title(title, fontsize=9)
    plt.axis('off')
plt.tight_layout()
plt.show()


## Optional caption generation

Use a GPU runtime. The model is loaded only for caption generation and should not be kept loaded during GeoDiff-GAN training. The prompt forces evidence-only, location-free remote-sensing captions.

In [ ]:
!pip install -q accelerate bitsandbytes qwen-vl-utils 'transformers>=4.57.0'

In [ ]:
import json
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, BitsAndBytesConfig
import transformers

CAPTION_MODEL = 'Qwen/Qwen3-VL-8B-Instruct'
CAPTION_OUTPUT = OUTPUT_ROOT / 'captions_qwen3vl.jsonl'
CAPTION_SPLIT = 'all'
CAPTION_LIMIT = None
PREFERRED_CAPTION = 'descriptive'
DISPLAY_GAIN = 2.5

CAPTION_PROMPT = '''You are captioning a Sentinel-2 RGB satellite image patch for prompt-conditioned super-resolution research.
Return strict JSON only. Do not wrap it in markdown.
Required schema:
{
  "brief": "one short evidence-only phrase, maximum 18 words",
  "descriptive": "one or two neutral sentences describing visible land cover, objects, density, terrain, and texture",
  "analytical": {
    "land_cover": ["visible land-cover classes only"],
    "visible_objects": ["visible object or structure categories only"],
    "object_density": "sparse | moderate | dense | mixed | unclear",
    "terrain": "flat | hilly | mountainous | coastal | riverine | arid | mixed | unclear",
    "texture": "smooth | fine-grained | coarse | grid-like | linear | mixed | unclear",
    "spatial_layout": "short description of dominant spatial arrangement",
    "uncertainty": "low | medium | high"
  }
}
Rules: only describe visible evidence; do not infer coordinates, city names, country names, ownership, people, events, or time. If uncertain, say unclear or possibly.'''

def resolve_model_class():
    for name in ('Qwen3VLForConditionalGeneration', 'AutoModelForMultimodalLM', 'AutoModelForVision2Seq'):
        cls = getattr(transformers, name, None)
        if cls is not None:
            return cls
    raise RuntimeError('Installed transformers does not expose a Qwen3-VL compatible class.')

def load_patch_image(relative_patch: str) -> Image.Image:
    with np.load(OUTPUT_ROOT / relative_patch) as data:
        arr = data['hr']
    if arr.shape[0] == 3:
        arr = arr.transpose(1, 2, 0)
    arr = (np.clip(arr * DISPLAY_GAIN, 0, 1) * 255).round().astype(np.uint8)
    return Image.fromarray(arr)

def parse_json_object(text: str) -> dict:
    stripped = text.strip()
    if stripped.startswith('```'):
        stripped = stripped.strip('`').removeprefix('json').strip()
    start, end = stripped.find('{'), stripped.rfind('}')
    if start >= 0 and end > start:
        try:
            value = json.loads(stripped[start:end+1])
            if isinstance(value, dict):
                return value
        except json.JSONDecodeError:
            pass
    return {'descriptive': text}

def analytical_to_text(value):
    if isinstance(value, str):
        return value
    if not isinstance(value, dict):
        return ''
    parts = []
    for key in ('land_cover', 'visible_objects'):
        item = value.get(key)
        if isinstance(item, list) and item:
            parts.append(f"{key.replace('_', ' ')}: " + ', '.join(map(str, item[:8])))
    for key in ('object_density', 'terrain', 'texture', 'spatial_layout', 'uncertainty'):
        item = value.get(key)
        if isinstance(item, str) and item.strip():
            parts.append(f"{key.replace('_', ' ')}: {item.strip()}")
    return '; '.join(parts)

def caption_text(payload, preferred='descriptive'):
    value = payload.get(preferred)
    text = analytical_to_text(value) if preferred == 'analytical' else value if isinstance(value, str) else ''
    if text:
        return text.strip()
    return payload.get('descriptive') or payload.get('brief') or analytical_to_text(payload.get('analytical')) or json.dumps(payload, ensure_ascii=True)

manifest_records = [json.loads(line) for line in (OUTPUT_ROOT / 'manifest_relative.jsonl').read_text(encoding='utf-8').splitlines() if line.strip()]
if CAPTION_SPLIT != 'all':
    manifest_records = [r for r in manifest_records if r['split'] == CAPTION_SPLIT]
if CAPTION_LIMIT is not None:
    manifest_records = manifest_records[:CAPTION_LIMIT]

completed = set()
if CAPTION_OUTPUT.exists():
    for line in CAPTION_OUTPUT.read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        try:
            completed.add(json.loads(line)['patch'])
        except json.JSONDecodeError:
            continue

quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
model_cls = resolve_model_class()
model = model_cls.from_pretrained(CAPTION_MODEL, device_map='auto', quantization_config=quant, dtype=torch.float16).eval()
processor = AutoProcessor.from_pretrained(CAPTION_MODEL)

CAPTION_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with CAPTION_OUTPUT.open('a', encoding='utf-8') as handle:
    for record in tqdm(manifest_records, desc='caption patches'):
        if record['patch'] in completed:
            continue
        image = load_patch_image(record['patch'])
        messages = [{'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': CAPTION_PROMPT}]}]
        inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors='pt').to(model.device)
        with torch.inference_mode():
            generated = model.generate(**inputs, max_new_tokens=220, do_sample=False)
        text = processor.batch_decode(generated[:, inputs['input_ids'].shape[-1]:], skip_special_tokens=True)[0].strip()
        payload = parse_json_object(text)
        handle.write(json.dumps({
            'patch': record['patch'],
            'tile_id': record['tile_id'],
            'caption': caption_text(payload, PREFERRED_CAPTION),
            'captions': payload,
            'preferred_caption': PREFERRED_CAPTION,
            'source_model': CAPTION_MODEL,
        }, ensure_ascii=True) + '\n')
        handle.flush()

print('Caption file:', CAPTION_OUTPUT)

In [ ]:
# Visualize generated captions with their HR patches.
import random
import matplotlib.pyplot as plt

caption_file = OUTPUT_ROOT / 'captions_qwen3vl.jsonl'
if not caption_file.exists():
    print('No caption file yet:', caption_file)
else:
    caption_records = [json.loads(line) for line in caption_file.read_text(encoding='utf-8').splitlines() if line.strip()]
    samples = random.sample(caption_records, min(6, len(caption_records)))
    plt.figure(figsize=(16, 4 * max(1, len(samples))))
    for i, record in enumerate(samples, 1):
        with np.load(OUTPUT_ROOT / record['patch']) as data:
            hr = data['hr']
        image = np.transpose(hr, (1, 2, 0)) if hr.shape[0] == 3 else hr
        captions = record.get('captions', {})
        text = (
            f"brief: {captions.get('brief', '')}\n"
            f"descriptive: {captions.get('descriptive', record.get('caption', ''))}\n"
            f"analytical: {json.dumps(captions.get('analytical', {}), ensure_ascii=True)[:260]}"
        )
        ax_img = plt.subplot(len(samples), 2, 2 * i - 1)
        ax_img.imshow(np.clip(image * VISUAL_DISPLAY_GAIN, 0, 1))
        ax_img.axis('off')
        ax_img.set_title(Path(record['patch']).name, fontsize=9)
        ax_txt = plt.subplot(len(samples), 2, 2 * i)
        ax_txt.axis('off')
        ax_txt.text(0, 1, text, va='top', wrap=True, fontsize=10)
    plt.tight_layout()
    plt.show()


In [ ]:
import shutil
zip_base = OUTPUT_ROOT.parent / OUTPUT_ROOT.name
archive = shutil.make_archive(str(zip_base), 'zip', OUTPUT_ROOT)
print('Created:', archive)

In [ ]:
# Kaggle helper: run after uploading geodiff_hr_dataset.zip as a Kaggle dataset.
from pathlib import Path
import json
from collections import Counter

KAGGLE_HR_DATASET_ROOT = Path('/kaggle/input/geodiff-hr-dataset')
RELATIVE_MANIFEST = KAGGLE_HR_DATASET_ROOT / 'manifest_relative.jsonl'
ABS_MANIFEST = Path('/kaggle/working/geodiff-gan-output/manifest.jsonl')
ABS_MANIFEST.parent.mkdir(parents=True, exist_ok=True)

records = []
for line in RELATIVE_MANIFEST.read_text(encoding='utf-8').splitlines():
    if line.strip():
        record = json.loads(line)
        record['patch'] = str((KAGGLE_HR_DATASET_ROOT / record['patch']).resolve())
        records.append(record)

with ABS_MANIFEST.open('w', encoding='utf-8') as handle:
    for record in records:
        handle.write(json.dumps(record, ensure_ascii=True) + '\n')

print('Wrote:', ABS_MANIFEST)
print('Patches:', len(records))
print('Splits:', Counter(record['split'] for record in records))
print('Tiles:', Counter(record['tile_id'] for record in records))
print('Captions path:', KAGGLE_HR_DATASET_ROOT / 'captions_qwen3vl.jsonl')

# Recommended GeoDiff-GAN config flags for your Kaggle training notebook:
# runtime_config['data']['manifest'] = str(ABS_MANIFEST)
# runtime_config['data']['captions'] = str(KAGGLE_HR_DATASET_ROOT / 'captions_qwen3vl.jsonl')
# runtime_config['data']['caption_field'] = 'caption'       # fixed default top-level caption
# runtime_config['data']['caption_sampling'] = 'random'     # train split only: random brief/descriptive/analytical
# runtime_config['data']['random_caption_fields'] = ['brief', 'descriptive', 'analytical']
# runtime_config['data']['train_degradation_sampling'] = 'random'  # new LR noise/degradation each train call
